In [81]:

import os
from pathlib import Path
import glob
import re

In [82]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession



In [83]:
BASE_DIR = "/volume/data/"

DIRECTORY_RAW = BASE_DIR+'despesas_contabeis/data/raw/'
DIRECTORY_SILVER = BASE_DIR+'despesas_contabeis/data/silver/'

In [84]:
spark = SparkSession.builder\
    .appName("Preprocessamento")\
        .config("spark.driver.memory", "8g") \
            .config("spark.executor.memory", "8g").getOrCreate()


In [85]:
def load_datas_spark(files):
    dffinal = spark.read.format("parquet").load(files[0])
    print(dffinal.printSchema())
    for f in files[1:]:
        print(f"Processando: {f}")
        try:
            results = spark.read.parquet(f)
            if "VL_SALDO_INICIAL" not in results.columns:
                results = results.withColumn("VL_SALDO_INICIAL", F.lit(0.0))
                results = results.select(*dffinal.columns)

            dffinal = dffinal.unionByName(results)
        except Exception as e:
            
            print(f"Erro ao processar {f}: {e}")
    return dffinal

In [86]:
files_path = glob.glob(DIRECTORY_RAW+'*.parquet')

In [87]:
#files_path_filter
len(files_path)

files_path.sort(reverse=True)

In [100]:

print("Qtd de files:", len(files_path))
df = load_datas_spark(files_path)

Qtd de files: 42
root
 |-- DATA: string (nullable = true)
 |-- REG_ANS: long (nullable = true)
 |-- CD_CONTA_CONTABIL: long (nullable = true)
 |-- DESCRICAO: string (nullable = true)
 |-- VL_SALDO_INICIAL: string (nullable = true)
 |-- VL_SALDO_FINAL: string (nullable = true)

None
Processando: /volume/data/despesas_contabeis/data/raw/4T2023.parquet
Processando: /volume/data/despesas_contabeis/data/raw/4T2022.parquet
Processando: /volume/data/despesas_contabeis/data/raw/4T2021.parquet
Processando: /volume/data/despesas_contabeis/data/raw/4T2020.parquet
Processando: /volume/data/despesas_contabeis/data/raw/4T2019.parquet
Processando: /volume/data/despesas_contabeis/data/raw/4T2018.parquet
Processando: /volume/data/despesas_contabeis/data/raw/4T2017.parquet
Processando: /volume/data/despesas_contabeis/data/raw/4T2016.parquet
Processando: /volume/data/despesas_contabeis/data/raw/4T2015.parquet
Processando: /volume/data/despesas_contabeis/data/raw/3T2024.parquet
Processando: /volume/data/d

In [101]:
df.count()

28873711

In [102]:
df.printSchema()

root
 |-- DATA: string (nullable = true)
 |-- REG_ANS: double (nullable = true)
 |-- CD_CONTA_CONTABIL: double (nullable = true)
 |-- DESCRICAO: string (nullable = true)
 |-- VL_SALDO_INICIAL: string (nullable = true)
 |-- VL_SALDO_FINAL: string (nullable = true)



In [103]:
df.show(5)

+----------+--------+-----------------+--------------------+----------------+--------------+
|      DATA| REG_ANS|CD_CONTA_CONTABIL|           DESCRICAO|VL_SALDO_INICIAL|VL_SALDO_FINAL|
+----------+--------+-----------------+--------------------+----------------+--------------+
|2024-10-01|420051.0|     4.71119017E8|ProvisÃ£o para De...|               0|             0|
|2024-10-01|420051.0|     4.71119018E8|     Outras Despesas|               0|             0|
|2024-10-01|420051.0|     4.71119019E8|(-) RecuperaÃ§Ã£o...|               0|             0|
|2024-10-01|420051.0|           4712.0|AJUSTES NEGATIVOS...|               0|             0|
|2024-10-01|420051.0|          47121.0|AJUSTES NEGATIVOS...|               0|             0|
+----------+--------+-----------------+--------------------+----------------+--------------+
only showing top 5 rows



In [104]:
df = (
    df
    .withColumn("REG_ANS", F.col("REG_ANS").cast("int"))


    .withColumn("CD_CONTA_CONTABIL", F.col("CD_CONTA_CONTABIL").cast('int').cast("string"))

    .withColumn("DATA", F.coalesce(
            F.to_date("DATA", "yyyy-MM-dd"),
            F.to_date("DATA", "MM/dd/yyyy"),
            F.to_date("DATA", "dd-MMM-yyyy"),
            F.to_date("DATA", "dd/MM/yyyy") # Example of another format
        ))


    .withColumn("VL_SALDO_INICIAL", F.regexp_replace("VL_SALDO_INICIAL", r"\.", ""))   
    .withColumn("VL_SALDO_INICIAL", F.regexp_replace("VL_SALDO_INICIAL", ",", "."))    
    .withColumn("VL_SALDO_INICIAL", F.col("VL_SALDO_INICIAL").cast("double"))

    .withColumn("VL_SALDO_FINAL", F.regexp_replace("VL_SALDO_FINAL", r"\.", ""))
    .withColumn("VL_SALDO_FINAL", F.regexp_replace("VL_SALDO_FINAL", ",", "."))
    .withColumn("VL_SALDO_FINAL", F.col("VL_SALDO_FINAL").cast("double"))
)

In [105]:
df.select(F.min('DATA').alias('min_data'), F.max('DATA').alias('max_data')).show()

+----------+----------+
|  min_data|  max_data|
+----------+----------+
|2015-01-01|2025-01-01|
+----------+----------+



In [106]:
df = df.withColumn('year_competencia', F.year(F.col('DATA')))

In [116]:
df.groupBy('year_competencia').count().orderBy('year_competencia').show()

+----------------+-------+
|year_competencia|  count|
+----------------+-------+
|            NULL| 728981|
|            2015|2094001|
|            2016|2104315|
|            2017|3088567|
|            2018|2463250|
|            2019|2863218|
|            2020|2903806|
|            2021|3062051|
|            2022|2447746|
|            2023|3194869|
|            2024|3061992|
|            2025| 705390|
+----------------+-------+



In [108]:
def corrigir_encoding(texto):
    """
    Corrige problemas comuns de encoding em textos portugueses
    """
    if any(re.findall('[\x80-\x90]', texto)) or ('Ã§' in texto or 'Ã£' in texto or 'Ã¡' in texto or 'Ã' in texto or 'Ã' in texto):
        try:
            #print("caracter especial encontrado")
            # Corrige dupla codificação (UTF-8 → Latin-1 → UTF-8)
            return texto.encode('latin-1').decode('utf-8').strip()
        except (UnicodeEncodeError, UnicodeDecodeError) as e:
            print(f"Erro ao corrigir '{texto}': {e}")
            try:
            # Se falhar, tenta outras abordagens
                return texto.encode('utf-8').decode('latin-1').strip()
            except:
                return texto.strip()
    #print("retorno sem tratamento")
    return texto.strip()

In [109]:
udf_corrigir_encoding = F.udf(corrigir_encoding, T.StringType())
df = df.withColumn('DESCRICAO', udf_corrigir_encoding(F.col('DESCRICAO')))

In [110]:

df.select(F.count(F.when(F.col('REG_ANS').isNull(), 'REG_ANS')).alias('REG_ANS')).show(5)
df = df.where(F.col('REG_ANS').isNotNull())

+-------+
|REG_ANS|
+-------+
| 155525|
+-------+



In [111]:
df.rdd.getNumPartitions()

140

In [112]:
df = df.repartition("DATA", "REG_ANS")

In [113]:
df.write.mode('overwrite').format('parquet').save(DIRECTORY_SILVER+'tb_union_dados_contabeis')

In [114]:
df = spark.read.parquet(DIRECTORY_SILVER+'tb_union_dados_contabeis')
df.count()

28718186

In [115]:
df.select('VL_SALDO_FINAL').describe().show()

+-------+--------------------+
|summary|      VL_SALDO_FINAL|
+-------+--------------------+
|  count|            28718186|
|   mean|   7604803.435033717|
| stddev|1.6601850795341715E8|
|    min|  -1.424000909082E10|
|    max|   3.827532591681E10|
+-------+--------------------+



In [117]:
files_path_operadoras = glob.glob(BASE_DIR+'operadoras/*')
len(files_path_operadoras)

2

In [118]:
dfoperadoras = spark.read.csv(BASE_DIR+'operadoras/', header=True, sep=',', encoding='utf-8', inferSchema=True)
print(dfoperadoras.count())
print(dfoperadoras.printSchema())

4138
root
 |-- _c0: integer (nullable = true)
 |-- Registro_ANS: string (nullable = true)
 |-- CNPJ: string (nullable = true)
 |-- Razao_Social: string (nullable = true)
 |-- Nome_Fantasia: string (nullable = true)
 |-- Modalidade: string (nullable = true)
 |-- Logradouro: string (nullable = true)
 |-- Numero: string (nullable = true)
 |-- Complemento: string (nullable = true)
 |-- Bairro: string (nullable = true)
 |-- Cidade: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- CEP: string (nullable = true)
 |-- DDD: string (nullable = true)
 |-- Telefone: string (nullable = true)
 |-- Fax: string (nullable = true)
 |-- Endereco_eletronico: string (nullable = true)
 |-- Representante: string (nullable = true)
 |-- Cargo_Representante: string (nullable = true)
 |-- Regiao_de_Comercializacao: string (nullable = true)
 |-- Data_Registro_ANS: string (nullable = true)
 |-- Data_Descredenciamento: string (nullable = true)
 |-- Motivo_do_Descredenciamento: string (nullable = true)

In [119]:
dfoperadoras.show(5)

+---+------------+--------------+--------------------+--------------------+--------------------+--------------------+------+-----------+------------+------------------+---+--------+---+--------+--------+--------------------+--------------------+--------------------+-------------------------+-----------------+----------------------+---------------------------+
|_c0|Registro_ANS|          CNPJ|        Razao_Social|       Nome_Fantasia|          Modalidade|          Logradouro|Numero|Complemento|      Bairro|            Cidade| UF|     CEP|DDD|Telefone|     Fax| Endereco_eletronico|       Representante| Cargo_Representante|Regiao_de_Comercializacao|Data_Registro_ANS|Data_Descredenciamento|Motivo_do_Descredenciamento|
+---+------------+--------------+--------------------+--------------------+--------------------+--------------------+------+-----------+------------+------------------+---+--------+---+--------+--------+--------------------+--------------------+--------------------+----------

In [120]:
print(dfoperadoras.columns)

['_c0', 'Registro_ANS', 'CNPJ', 'Razao_Social', 'Nome_Fantasia', 'Modalidade', 'Logradouro', 'Numero', 'Complemento', 'Bairro', 'Cidade', 'UF', 'CEP', 'DDD', 'Telefone', 'Fax', 'Endereco_eletronico', 'Representante', 'Cargo_Representante', 'Regiao_de_Comercializacao', 'Data_Registro_ANS', 'Data_Descredenciamento', 'Motivo_do_Descredenciamento']


In [121]:
dfoperadoras = dfoperadoras.select('Registro_ANS', 'Razao_Social', 'Nome_Fantasia', 'Modalidade', 'UF',  'Data_Registro_ANS', 'Data_Descredenciamento', 'Motivo_do_Descredenciamento')

In [122]:
dfjoin = df.join(dfoperadoras, df.REG_ANS == dfoperadoras.Registro_ANS, 'left')
dfjoin.groupBy('Modalidade').count().orderBy(F.desc('count')).show(10)

+--------------------+--------+
|          Modalidade|   count|
+--------------------+--------+
|  Cooperativa Médica|10254027|
|   Medicina de Grupo| 7633472|
|Administradora de...| 3413365|
|          Autogestão| 2733375|
|Odontologia de Grupo| 2284897|
|Cooperativa odont...| 1141821|
|         Filantropia|  988434|
|Seguradora Especi...|  268795|
+--------------------+--------+



In [123]:
dfjoin.count()

28718186

In [124]:
dfjoin.show(10)

+----------+-------+-----------------+--------------------+----------------+--------------+----------------+------------+--------------------+------------------+-----------------+---+-----------------+----------------------+---------------------------+
|      DATA|REG_ANS|CD_CONTA_CONTABIL|           DESCRICAO|VL_SALDO_INICIAL|VL_SALDO_FINAL|year_competencia|Registro_ANS|        Razao_Social|     Nome_Fantasia|       Modalidade| UF|Data_Registro_ANS|Data_Descredenciamento|Motivo_do_Descredenciamento|
+----------+-------+-----------------+--------------------+----------------+--------------+----------------+------------+--------------------+------------------+-----------------+---+-----------------+----------------------+---------------------------+
|2024-10-01| 420158|                1|               ATIVO|   9.426655864E7| 9.272141525E7|            2024|      420158|ASSOCIAÇÃO PADRE ...|PADRE ALBINO SAUDE|Medicina de Grupo| SP|       2016-07-06|                  NULL|                 

In [125]:
dfjoin = dfjoin.withColumn('size_cd_contabil', F.length(F.col('CD_CONTA_CONTABIL')))

In [126]:
dfjoin.groupBy('size_cd_contabil').count().orderBy(F.desc('count')).show(10)

+----------------+--------+
|size_cd_contabil|   count|
+----------------+--------+
|               9|11162643|
|               8| 5437601|
|               6| 3462536|
|               5| 3305898|
|               4| 2943027|
|               3| 1604856|
|               2|  607844|
|               1|  193781|
+----------------+--------+



In [127]:
dfjoin.groupBy('Registro_ANS', 'Razao_Social').agg(F.countDistinct('size_cd_contabil').alias('qtd_sizes')).groupBy('qtd_sizes').count().orderBy(F.desc('count')).show(10)

+---------+-----+
|qtd_sizes|count|
+---------+-----+
|        8| 1544|
+---------+-----+



In [128]:
dfplano_contas_a = dfjoin.select(F.col('CD_CONTA_CONTABIL').alias('cd_contabil'), F.col('DESCRICAO').alias('ds_conta_contabil'), 'size_cd_contabil').distinct()

dfplano_contas_a.write.format('parquet').mode('overwrite').save(BASE_DIR+'despesas_contabeis/data/raw/tb_plano_contas_bruto')

dfplano_contas_a = spark.read.format('parquet').load(BASE_DIR+'despesas_contabeis/data/raw/tb_plano_contas_bruto')

In [129]:

dfplano_contas_b = (dfplano_contas_a.where(F.col('size_cd_contabil') == 9)
                    .withColumn('n1_cd_contabil', F.col('cd_contabil').substr(1, 1))
                    .withColumn('n2_cd_contabil', F.col('cd_contabil').substr(1, 2))
                    .withColumn('n3_cd_contabil', F.col('cd_contabil').substr(1, 3))
                    .withColumn('n4_cd_contabil', F.col('cd_contabil').substr(1, 4))
                    .withColumn('n5_cd_contabil', F.col('cd_contabil').substr(1, 5))
                    .withColumn('n6_cd_contabil', F.col('cd_contabil').substr(1, 6))
                    .withColumn('n8_cd_contabil', F.col('cd_contabil').substr(1, 8))
                    )


dfplano_contas_b = (
    dfplano_contas_b
    .join(dfplano_contas_a.select(F.col('cd_contabil').alias('n1_cd_contabil_'), F.col('ds_conta_contabil').alias('n1_ds_conta_contabil')), dfplano_contas_b.n1_cd_contabil == F.col('n1_cd_contabil_'), 'left').drop('n1_cd_contabil_')
    .join(dfplano_contas_a.select(F.col('cd_contabil').alias('n2_cd_contabil_'), F.col('ds_conta_contabil').alias('n2_ds_conta_contabil')), dfplano_contas_b.n2_cd_contabil == F.col('n2_cd_contabil_'), 'left').drop('n2_cd_contabil_')
    .join(dfplano_contas_a.select(F.col('cd_contabil').alias('n3_cd_contabil_'), F.col('ds_conta_contabil').alias('n3_ds_conta_contabil')), dfplano_contas_b.n3_cd_contabil == F.col('n3_cd_contabil_'), 'left').drop('n3_cd_contabil_')
    .join(dfplano_contas_a.select(F.col('cd_contabil').alias('n4_cd_contabil_'), F.col('ds_conta_contabil').alias('n4_ds_conta_contabil')), dfplano_contas_b.n4_cd_contabil == F.col('n4_cd_contabil_'), 'left').drop('n4_cd_contabil_')
    .join(dfplano_contas_a.select(F.col('cd_contabil').alias('n5_cd_contabil_'), F.col('ds_conta_contabil').alias('n5_ds_conta_contabil')), dfplano_contas_b.n5_cd_contabil == F.col('n5_cd_contabil_'), 'left').drop('n5_cd_contabil_')
    .join(dfplano_contas_a.select(F.col('cd_contabil').alias('n6_cd_contabil_'), F.col('ds_conta_contabil').alias('n6_ds_conta_contabil')), dfplano_contas_b.n6_cd_contabil == F.col('n6_cd_contabil_'), 'left').drop('n6_cd_contabil_')
    .join(dfplano_contas_a.select(F.col('cd_contabil').alias('n8_cd_contabil_'), F.col('ds_conta_contabil').alias('n8_ds_conta_contabil')), dfplano_contas_b.n8_cd_contabil == F.col('n8_cd_contabil_'), 'left').drop('n8_cd_contabil_')
                    )

In [130]:
dfplano_contas_b.show(5)

+-----------+--------------------+----------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|cd_contabil|   ds_conta_contabil|size_cd_contabil|n1_cd_contabil|n2_cd_contabil|n3_cd_contabil|n4_cd_contabil|n5_cd_contabil|n6_cd_contabil|n8_cd_contabil|n1_ds_conta_contabil|n2_ds_conta_contabil|n3_ds_conta_contabil|n4_ds_conta_contabil|n5_ds_conta_contabil|n6_ds_conta_contabil|n8_ds_conta_contabil|
+-----------+--------------------+----------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|  126119031|CrÃ©dito de Previ...|               9|             1|            12|       

In [131]:
print(dfplano_contas_b.columns)

['cd_contabil', 'ds_conta_contabil', 'size_cd_contabil', 'n1_cd_contabil', 'n2_cd_contabil', 'n3_cd_contabil', 'n4_cd_contabil', 'n5_cd_contabil', 'n6_cd_contabil', 'n8_cd_contabil', 'n1_ds_conta_contabil', 'n2_ds_conta_contabil', 'n3_ds_conta_contabil', 'n4_ds_conta_contabil', 'n5_ds_conta_contabil', 'n6_ds_conta_contabil', 'n8_ds_conta_contabil']


In [132]:
dfplano_contas_b = dfplano_contas_b.select('n1_cd_contabil', 'n1_ds_conta_contabil', 'n2_cd_contabil', 'n2_ds_conta_contabil', 'n3_cd_contabil', 'n3_ds_conta_contabil', 
                                           'n4_cd_contabil', 'n4_ds_conta_contabil', 'n5_cd_contabil', 'n5_ds_conta_contabil', 'n6_cd_contabil', 'n6_ds_conta_contabil', 
                                           'n8_cd_contabil', 'n8_ds_conta_contabil', 'cd_contabil', 'ds_conta_contabil')

In [133]:
dfplano_contas_b.show(5)

+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+-----------+--------------------+
|n1_cd_contabil|n1_ds_conta_contabil|n2_cd_contabil|n2_ds_conta_contabil|n3_cd_contabil|n3_ds_conta_contabil|n4_cd_contabil|n4_ds_conta_contabil|n5_cd_contabil|n5_ds_conta_contabil|n6_cd_contabil|n6_ds_conta_contabil|n8_cd_contabil|n8_ds_conta_contabil|cd_contabil|   ds_conta_contabil|
+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+-----------+--------------------+
|             1|               ATIVO|            12|    ATIVO CIRCULANTE|           126|CRÉDITOS TRIBUTÁR...|          1261|CRÉDITOS TRIBUT

In [134]:
dfplano_contas_b.groupBy('cd_contabil', 'ds_conta_contabil').count().orderBy(F.desc('count')).show(10)

+-----------+--------------------+-----+
|cd_contabil|   ds_conta_contabil|count|
+-----------+--------------------+-----+
|  443119211|Despesas Operacio...|  243|
|  443119712|Despesas Operacio...|  243|
|  333119011|Receitas Operacio...|  243|
|  443119112|Despesas Operacio...|  243|
|  443119711|Despesas Operacio...|  243|
|  333119012|Receitas Operacio...|  243|
|  443119011|Despesas Operacio...|  243|
|  443119011|Despesas Operacio...|  243|
|  443119211|Despesas Operacio...|  243|
|  443119711|Despesas Operacio...|  243|
+-----------+--------------------+-----+
only showing top 10 rows



In [135]:
dfplano_contas_b = dfplano_contas_b.distinct()

In [136]:
dfplano_contas_b = dfplano_contas_b.dropDuplicates(subset = ['cd_contabil'])

In [137]:
dfplano_contas_b.write.format('parquet').mode('overwrite').save(DIRECTORY_SILVER+'/tb_plano_contas')

In [138]:
dfplano_contas_b = spark.read.format('parquet').load(DIRECTORY_SILVER+'/tb_plano_contas')

In [139]:
dfjoin = dfjoin.where(F.col('size_cd_contabil') == 9)

In [140]:
dfjoin = dfjoin.join(
    dfplano_contas_b, dfjoin.CD_CONTA_CONTABIL == dfplano_contas_b.cd_contabil, 'left'
)

In [141]:
dfjoin.show(5)

+----------+-------+-----------------+--------------------+----------------+--------------+----------------+------------+--------------------+------------------+-----------------+---+-----------------+----------------------+---------------------------+----------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+--------------+--------------------+-----------+--------------------+
|      DATA|REG_ANS|CD_CONTA_CONTABIL|           DESCRICAO|VL_SALDO_INICIAL|VL_SALDO_FINAL|year_competencia|Registro_ANS|        Razao_Social|     Nome_Fantasia|       Modalidade| UF|Data_Registro_ANS|Data_Descredenciamento|Motivo_do_Descredenciamento|size_cd_contabil|n1_cd_contabil|n1_ds_conta_contabil|n2_cd_contabil|n2_ds_conta_contabil|n3_cd_contabil|n3_ds_conta_contabil|n4_cd_contabil|n4_ds_conta_contabil|n5_cd_contabil|n5_ds_conta_conta

In [142]:
dfjoin = dfjoin.drop(*['CD_CONTA_CONTABIL', 'DESCRICAO', 'size_cd_contabil'])

In [143]:
dfjoin = dfjoin.distinct()
dfjoin.count()

10947574

In [144]:
dfjoin.write.mode('overwrite').format('parquet').save(DIRECTORY_SILVER+'tb_dados_contabeis_operadoras')

In [145]:
dfjoin = spark.read.format('parquet').load(DIRECTORY_SILVER+'tb_dados_contabeis_operadoras')